# Brain Tumor MRI Classification with ResNet18 - Report Version

This version keeps the core training pipeline and the three visuals useful for reporting: class distribution, training curves, and prediction examples.

Extra preview/comparison cells were removed to make the notebook easier to explain.


## 0. Environment Setup

Install lightweight packages and download the Kaggle dataset.


In [ ]:
import importlib.util
import os
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path

LIGHTWEIGHT_PACKAGES = [
    ("kagglehub", "kagglehub"),
    ("tqdm", "tqdm"),
]

for package_name, module_name in LIGHTWEIGHT_PACKAGES:
    if importlib.util.find_spec(module_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_name])

import kagglehub

DATASET_SLUG = "navoneel/brain-mri-images-for-brain-tumor-detection"
DATASET_ROOT = Path(kagglehub.dataset_download(DATASET_SLUG))

# TODO: Locate the brain_tumor_dataset folder under DATASET_ROOT.
DATA_DIR = DATASET_ROOT / "brain_tumor_dataset"  # Update this path as needed

# Small non-core helper: choose a writable work directory in Colab or locally.
colab_root = Path("/content")
WORK_DIR = colab_root / "brain_tumor_resnet_work" if colab_root.exists() and os.access(colab_root, os.W_OK) else Path(tempfile.gettempdir()) / "brain_tumor_resnet_work"

# Keep this simple version safe: create the work folder, but do not delete it automatically.
WORK_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORK_DIR)

print("Dataset root:", DATASET_ROOT)
print("Working directory:", WORK_DIR)
print("TODO: set DATA_DIR to the folder containing yes/ and no/")


## 1. Imports and Random Seed

Import libraries and set a fixed random seed so the experiment is easier to reproduce.


In [ ]:
import random
import time
import copy

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms, models
from torchvision.datasets import ImageFolder
from PIL import Image
from tqdm.notebook import tqdm

RANDOM_SEED = 47
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

%matplotlib inline
print("Libraries imported.")
print("PyTorch version:", torch.__version__)


## 2. Dataset Check

Confirm that the dataset contains the two folders `yes/` and `no/`.


In [ ]:
# TODO: Set DATA_DIR if you did not complete it in Section 0.
# Example shape, not the answer:
# DATA_DIR = Path("...") / "brain_tumor_dataset"

assert DATA_DIR is not None, "Set DATA_DIR before continuing."
assert (DATA_DIR / "yes").exists(), "Missing yes/ folder."
assert (DATA_DIR / "no").exists(), "Missing no/ folder."

for class_name in ["yes", "no"]:
    files = [p for p in (DATA_DIR / class_name).iterdir() if p.is_file() and not p.name.startswith(".")]
    print(f"{class_name}: {len(files)} images")


## 3. Data Augmentation Function

Define a helper function that creates extra training images by rotating, shifting, and flipping originals.


In [ ]:
def data_augment(img_dir, n_samples, save_to_dir):
    """Create augmented copies of images from img_dir and save them to save_to_dir."""
    img_dir = Path(img_dir)
    save_to_dir = Path(save_to_dir)
    save_to_dir.mkdir(parents=True, exist_ok=True)

    # Scaffold: define a torchvision transform pipeline here.
    # TODO: add realistic image augmentations.
    augment = transforms.Compose([
        transforms.RandomRotation(15),
        transforms.RandomAffine(
            degrees=0,
            translate=(0.05, 0.05),
            shear=5,
            scale=(0.95, 1.05),
        ),
        transforms.RandomHorizontalFlip(p=0.5),
    ])

    image_files = sorted([
        p for p in img_dir.iterdir()
        if p.is_file() and not p.name.startswith(".")
    ])

    # Save one copy of the original plus n_samples augmented copies per image.
    # This keeps the folder usable even if an augmentation produces an unusual result.
    for file_idx, img_path in enumerate(image_files):
        try:
            img = Image.open(img_path).convert("RGB")
        except Exception as exc:
            print(f"Skipping unreadable image: {img_path} ({exc})")
            continue

        original_path = save_to_dir / f"{img_path.stem}_orig{img_path.suffix}"
        img.save(original_path)

        # TODO: loop over files in img_dir, open each image with PIL, create n_samples
        # augmented copies, and save them to save_to_dir.
        for aug_idx in range(n_samples):
            aug_img = augment(img)
            save_path = save_to_dir / f"{img_path.stem}_aug_{aug_idx:02d}.jpg"
            aug_img.save(save_path)

    print(f"Saved augmented images to {save_to_dir}: {len(list(save_to_dir.glob('*')))} files")

## 4. Build Augmented Dataset

Create the `augmented-images` folder and generate augmented images for both classes.


In [ ]:
AUG_DIR = Path("augmented-images")
if AUG_DIR.exists():
    shutil.rmtree(AUG_DIR)

for class_name in ["yes", "no"]:
    (AUG_DIR / class_name).mkdir(parents=True, exist_ok=True)

# TODO: call data_augment once for yes and once for no.
# Suggested starting point: 6 augmented samples per original image.
N_AUG_PER_IMAGE = 6
data_augment(DATA_DIR / "yes", n_samples=N_AUG_PER_IMAGE, save_to_dir=AUG_DIR / "yes")
data_augment(DATA_DIR / "no", n_samples=N_AUG_PER_IMAGE, save_to_dir=AUG_DIR / "no")

# After augmentation, use this as the active data directory.
data_dir = AUG_DIR

print("Active data directory:", data_dir)
print("yes:", len(list((data_dir / "yes").glob("*"))))
print("no: ", len(list((data_dir / "no").glob("*"))))

## 5. Split Data into Train / Val / Test

Copy images into three folders: training, validation, and testing.


In [ ]:
SPLITS = ["TRAIN", "VAL", "TEST"]
CLASSES = ["YES", "NO"]

for split in SPLITS:
    split_path = Path(split)
    if split_path.exists():
        shutil.rmtree(split_path)
    for class_name in CLASSES:
        Path(split, class_name).mkdir(parents=True, exist_ok=True)

# TODO: iterate through AUG_DIR/yes and AUG_DIR/no.
# TODO: shuffle files with RANDOM_SEED.
# TODO: copy each file into TEST, TRAIN, or VAL.
TEST_RATIO = 0.15
VAL_RATIO = 0.15

random.seed(RANDOM_SEED)

class_source_map = {
    "YES": data_dir / "yes",
    "NO": data_dir / "no",
}

for class_name, src_dir in class_source_map.items():
    files = sorted([
        p for p in src_dir.iterdir()
        if p.is_file() and not p.name.startswith(".")
    ])
    random.shuffle(files)

    n_total = len(files)
    n_test = int(round(n_total * TEST_RATIO))
    n_val = int(round(n_total * VAL_RATIO))

    test_files = files[:n_test]
    val_files = files[n_test:n_test + n_val]
    train_files = files[n_test + n_val:]

    split_map = {
        "TRAIN": train_files,
        "VAL": val_files,
        "TEST": test_files,
    }

    for split, split_files in split_map.items():
        for src in split_files:
            dst = Path(split) / class_name / src.name
            shutil.copy2(src, dst)

# Non-core helper for checking your result:
def count_images(folder):
    return sum(1 for p in Path(folder).rglob("*") if p.is_file())

for split in SPLITS:
    print(split, count_images(split))
    for class_name in CLASSES:
        print(" ", class_name, count_images(Path(split) / class_name))

## 6. Load Images into Python Arrays

Read images from folders into `X` arrays and convert folder names into numeric labels `y`.


In [ ]:
IMG_SIZE = (224, 224)


def load_data(dir_path, img_size=IMG_SIZE):
    """Load images from an ImageFolder-style directory."""
    dir_path = Path(dir_path)
    X, y, labels = [], [], {}

    # TODO: iterate over sorted class folders.
    # TODO: read, convert, resize, and append each image.
    # TODO: assign integer labels consistently.
    class_dirs = sorted([p for p in dir_path.iterdir() if p.is_dir()])

    for label_id, class_dir in enumerate(class_dirs):
        labels[class_dir.name] = label_id

        image_files = sorted([
            p for p in class_dir.iterdir()
            if p.is_file() and not p.name.startswith(".")
        ])

        for img_path in image_files:
            try:
                img = Image.open(img_path).convert("RGB")
                img = img.resize(img_size)
            except Exception as exc:
                print("Skipping unreadable image:", img_path, exc)
                continue

            X.append(np.asarray(img, dtype=np.uint8))
            y.append(label_id)

    return np.asarray(X, dtype=np.uint8), np.asarray(y, dtype=np.int64), labels


# TODO: load TRAIN, VAL, and TEST.
X_train, y_train, labels = load_data("TRAIN")
X_val, y_val, _ = load_data("VAL")
X_test, y_test, _ = load_data("TEST")

print("Labels:", labels)
print("X_train:", X_train.shape, X_train.dtype)
print("X_val:  ", X_val.shape, X_val.dtype)
print("X_test: ", X_test.shape, X_test.dtype)
print("y_train:", y_train.shape, np.bincount(y_train))
print("y_val:  ", y_val.shape, np.bincount(y_val))
print("y_test: ", y_test.shape, np.bincount(y_test))


## 7. Class Distribution Bar Chart

Show how many `YES` and `NO` images are in train, validation, and test sets. This is useful for reporting whether the split is balanced.


In [ ]:
# TODO: count labels in y_train, y_val, and y_test.
# TODO: create a bar chart comparing class counts across splits.

# Helpful NumPy hint:
# np.sum(y_train == 0)
id_to_label = {label_id: class_name for class_name, label_id in labels.items()}
class_ids = sorted(id_to_label.keys())
split_names = ["Train", "Val", "Test"]
y_sets = [y_train, y_val, y_test]

counts = {
    id_to_label[class_id]: [int(np.sum(y_split == class_id)) for y_split in y_sets]
    for class_id in class_ids
}

x = np.arange(len(split_names))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 5))
for offset, (class_name, class_counts) in zip([-width / 2, width / 2], counts.items()):
    bars = ax.bar(x + offset, class_counts, width, label=class_name)
    ax.bar_label(bars, padding=3)

ax.set_xticks(x)
ax.set_xticklabels(split_names)
ax.set_ylabel("Sample Count")
ax.set_title("Class Distribution Across Train / Val / Test")
ax.legend()
plt.tight_layout()
plt.show()

counts

## 8. Simple Preprocessing

Resize every image to the same size so the model can read them.


In [ ]:
def crop_imgs(set_name, add_pixels_value=0, img_size=IMG_SIZE):
    """Simplified preprocessing: keep the image content and resize it."""
    processed = []

    for img in set_name:
        # TODO: implement the preprocessing steps.
        # Simple version: no contour cropping; just make every image RGB uint8 and 224x224.
        arr = np.asarray(img)

        if arr.max() <= 1.0:
            arr = arr * 255

        arr = np.clip(arr, 0, 255).astype(np.uint8)

        if arr.ndim == 2:
            arr = np.stack([arr, arr, arr], axis=-1)
        elif arr.ndim == 3 and arr.shape[-1] == 1:
            arr = np.repeat(arr, 3, axis=-1)

        resized_img = Image.fromarray(arr).resize(img_size)
        processed.append(np.asarray(resized_img, dtype=np.uint8))

    # TODO: return the list of processed images.
    return np.asarray(processed, dtype=np.uint8)


# TODO: preprocess train/val/test images.
X_train_crop = crop_imgs(X_train)
X_val_crop = crop_imgs(X_val)
X_test_crop = crop_imgs(X_test)

print("X_train_crop:", X_train_crop.shape, X_train_crop.dtype, X_train_crop.min(), X_train_crop.max())
print("X_val_crop:  ", X_val_crop.shape, X_val_crop.dtype)
print("X_test_crop: ", X_test_crop.shape, X_test_crop.dtype)


## 9. Save Processed Images

Save processed arrays back into folders so `ImageFolder` can load them.


In [ ]:
CROP_SPLITS = ["TRAIN_CROP", "VAL_CROP", "TEST_CROP"]
for split in CROP_SPLITS:
    split_path = Path(split)
    if split_path.exists():
        shutil.rmtree(split_path)
    for class_name in CLASSES:
        Path(split, class_name).mkdir(parents=True, exist_ok=True)


def save_new_images(x_set, y_set, folder_name):
    """Save RGB images into ImageFolder-style class directories."""
    folder_name = Path(folder_name)

    # TODO: save each image into YES or NO according to its label.
    # TODO: convert the NumPy array back to a PIL image before saving.
    id_to_label = {label_id: class_name for class_name, label_id in labels.items()}

    for i, (img, label) in enumerate(zip(x_set, y_set)):
        class_name = id_to_label[int(label)]
        save_dir = folder_name / class_name
        save_dir.mkdir(parents=True, exist_ok=True)

        img_uint8 = np.clip(img, 0, 255).astype(np.uint8)
        save_path = save_dir / f"{i:05d}.jpg"
        Image.fromarray(img_uint8).save(save_path)


# TODO: save processed train/val/test images.
save_new_images(X_train_crop, y_train, "TRAIN_CROP")
save_new_images(X_val_crop, y_val, "VAL_CROP")
save_new_images(X_test_crop, y_test, "TEST_CROP")

for split in CROP_SPLITS:
    print(split, count_images(split))


## 10. Define ResNet Input Transforms（IMAGENET_MEAN 在这里）

这一节定义 ResNet18 读取图片前要做的处理：resize、训练增强、转 Tensor、按 ImageNet mean/std 标准化。


In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

# TODO: define train_transform.
# TODO: optionally define eval_transform separately.

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.10, contrast=0.10),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

print("Transforms ready.")

## 11. Create Dataset and DataLoader

这一节才是 PyTorch 的数据集和批量读取：`ImageFolder` 负责按文件夹读图片，`DataLoader` 负责一批一批送进模型。


In [ ]:
BATCH_SIZE = 8

# TODO: create ImageFolder datasets.
# train_dataset = ImageFolder("TRAIN_CROP", transform=train_transform)
# val_dataset = ImageFolder("VAL_CROP", transform=...)
# test_dataset = ImageFolder("TEST_CROP", transform=...)
train_dataset = ImageFolder("TRAIN_CROP", transform=train_transform)
val_dataset = ImageFolder("VAL_CROP", transform=eval_transform)
test_dataset = ImageFolder("TEST_CROP", transform=eval_transform)

# TODO: create DataLoader objects.
# train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
# val_dataloader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_dataloader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

dataloaders = {
    "train": train_dataloader,
    "val": val_dataloader,
    "test": test_dataloader,
}

# TODO: define class_names and dataset_sizes.
class_names = train_dataset.classes
dataset_sizes = {
    "train": len(train_dataset),
    "val": len(val_dataset),
    "test": len(test_dataset),
}

print("class_names:", class_names)
print("dataset_sizes:", dataset_sizes)

sample_inputs, sample_labels = next(iter(train_dataloader))
print("one batch image tensor:", sample_inputs.shape)
print("one batch labels:", sample_labels.shape)


## 12. Training Function

Define the repeated training and validation process.


In [ ]:
def train_model(model, criterion, optimizer, num_epochs=3):
    """Train a model and return the best validation checkpoint."""
    since = time.time()
    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0

    history = {
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": [],
    }

    # TODO: implement the epoch loop and train/val phases.
    # TODO: compute loss and accuracy for each phase.
    # TODO: keep the best validation model weights.
    for epoch in range(num_epochs):
        print(f"Epoch {epoch + 1}/{num_epochs}")
        print("-" * 30)

        for phase in ["train", "val"]:
            if phase == "train":
                model.train()
            else:
                model.eval()

            running_loss = 0.0
            running_corrects = 0

            for inputs, batch_labels in tqdm(dataloaders[phase], desc=phase, leave=False):
                inputs = inputs.to(device)
                batch_labels = batch_labels.to(device)

                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == "train"):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, batch_labels)

                    if phase == "train":
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == batch_labels.data).item()

            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc = running_corrects / dataset_sizes[phase]

            history[f"{phase}_loss"].append(epoch_loss)
            history[f"{phase}_acc"].append(epoch_acc)

            print(f"{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}")

            if phase == "val" and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_model_wts = copy.deepcopy(model.state_dict())

        print()

    elapsed = time.time() - since
    print(f"Training complete in {elapsed // 60:.0f}m {elapsed % 60:.0f}s")
    print(f"Best val Acc: {best_acc:.4f}")

    model.load_state_dict(best_model_wts)
    return model, history

## 13. Build ResNet18 Model

Load a pre-trained ResNet18 model and replace the final layer for two classes.


In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# TODO: load a ResNet18 model.
try:
    weights = models.ResNet18_Weights.DEFAULT
    model_ft = models.resnet18(weights=weights)
    print("Loaded ImageNet pretrained ResNet18 weights.")
except Exception as exc:
    print("Could not download pretrained weights. Using random initialization.")
    print("Reason:", exc)
    model_ft = models.resnet18(weights=None)

# TODO: freeze backbone parameters.
for param in model_ft.parameters():
    param.requires_grad = False

# TODO: replace the final fully connected layer.
num_ftrs = model_ft.fc.in_features
model_ft.fc = nn.Linear(num_ftrs, len(class_names))

# TODO: move the model to device.
model_ft = model_ft.to(device)

# TODO: define criterion and optimizer.
criterion = nn.CrossEntropyLoss()
optimizer_ft = optim.Adam(model_ft.fc.parameters(), lr=1e-3)

print(model_ft.fc)


## 14. Train the Model and Plot Curves

Train the model, then show loss and accuracy curves. These curves help explain whether the model is learning.


In [ ]:
NUM_EPOCHS = 3

# TODO: call train_model.
# model_ft = train_model(model_ft, criterion, optimizer_ft, num_epochs=NUM_EPOCHS)
model_ft, history = train_model(
    model_ft,
    criterion,
    optimizer_ft,
    num_epochs=NUM_EPOCHS,
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history["train_loss"], label="train")
axes[0].plot(history["val_loss"], label="val")
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()

axes[1].plot(history["train_acc"], label="train")
axes[1].plot(history["val_acc"], label="val")
axes[1].set_title("Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].legend()

plt.tight_layout()
plt.show()

## 15. Visualize Prediction Results by Class

Show validation examples from each class, so the final display includes both `YES` and `NO` cases.


In [ ]:
def imshow(inp, title=None):
    """Display a normalized image tensor after reversing ImageNet normalization."""
    # TODO: convert tensor to HWC NumPy format.
    inp = inp.detach().cpu().numpy().transpose((1, 2, 0))

    # TODO: unnormalize with IMAGENET_MEAN and IMAGENET_STD.
    mean = np.array(IMAGENET_MEAN)
    std = np.array(IMAGENET_STD)
    inp = std * inp + mean

    # TODO: clip values to [0, 1] and display.
    inp = np.clip(inp, 0, 1)
    plt.imshow(inp)
    if title is not None:
        plt.title(title)
    plt.axis("off")


def visualize_model_by_class(model, dataloader, images_per_class=3):
    """Show validation images from each true class with model predictions."""
    # TODO: collect a few examples for each class and display predictions.
    was_training = model.training
    model.eval()
    examples = {class_idx: [] for class_idx in range(len(class_names))}

    with torch.no_grad():
        for inputs, batch_labels in dataloader:
            inputs = inputs.to(device)
            batch_labels = batch_labels.to(device)

            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)

            for j in range(inputs.size(0)):
                true_idx = int(batch_labels[j].item())
                pred_idx = int(preds[j].item())

                if len(examples[true_idx]) < images_per_class:
                    examples[true_idx].append((inputs[j].cpu(), pred_idx, true_idx))

            if all(len(v) >= images_per_class for v in examples.values()):
                break

    n_classes = len(class_names)
    fig, axes = plt.subplots(n_classes, images_per_class, figsize=(4 * images_per_class, 3.4 * n_classes))
    axes = np.asarray(axes).reshape(n_classes, images_per_class)

    for class_idx in range(n_classes):
        for col in range(images_per_class):
            ax = axes[class_idx, col]
            ax.axis("off")

            if col >= len(examples[class_idx]):
                ax.set_title(f"true: {class_names[class_idx]} | no sample")
                continue

            img_tensor, pred_idx, true_idx = examples[class_idx][col]
            plt.sca(ax)
            imshow(img_tensor)
            ax.set_title(f"pred: {class_names[pred_idx]} | true: {class_names[true_idx]}")

    model.train(mode=was_training)
    plt.tight_layout()
    plt.show()


# TODO: call visualize_model_by_class(model_ft).
visualize_model_by_class(model_ft, val_dataloader, images_per_class=3)


## 16. Confusion Matrix on Test Set

Create a standard 2x2 confusion matrix with `TP`, `FP`, `FN`, and `TN`.


In [ ]:
# TODO: compute and display a standard TP / FP / FN / TN confusion matrix.
def compute_confusion_counts(model, dataloader, positive_label="YES"):
    """Return TP, FP, FN, TN counts for the positive class."""
    was_training = model.training
    model.eval()

    positive_idx = class_names.index(positive_label)
    tp = fp = fn = tn = 0

    with torch.no_grad():
        for inputs, batch_labels in dataloader:
            inputs = inputs.to(device)
            batch_labels = batch_labels.to(device)

            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)

            for true_idx, pred_idx in zip(batch_labels.cpu().numpy(), preds.cpu().numpy()):
                true_positive = int(true_idx) == positive_idx
                pred_positive = int(pred_idx) == positive_idx

                if true_positive and pred_positive:
                    tp += 1
                elif (not true_positive) and pred_positive:
                    fp += 1
                elif true_positive and (not pred_positive):
                    fn += 1
                else:
                    tn += 1

    model.train(mode=was_training)
    return tp, fp, fn, tn


TP, FP, FN, TN = compute_confusion_counts(model_ft, test_dataloader, positive_label="YES")

print("Positive class: YES")
print("TP:", TP, "FP:", FP, "FN:", FN, "TN:", TN)

# Draw a clean 2x2 confusion matrix manually to avoid table text overlap.
fig, ax = plt.subplots(figsize=(7.2, 5.8))
ax.set_xlim(0, 2)
ax.set_ylim(0, 2)
ax.set_aspect("equal")
ax.axis("off")

cells = [
    # x, y, short name, long name, count, color
    (0, 1, "TP", "True Positive", TP, "#C8E6C9"),
    (1, 1, "FN", "False Negative", FN, "#FFE0B2"),
    (0, 0, "FP", "False Positive", FP, "#FFCDD2"),
    (1, 0, "TN", "True Negative", TN, "#BBDEFB"),
]

for x, y, short_name, long_name, count, color in cells:
    rect = plt.Rectangle((x, y), 1, 1, facecolor=color, edgecolor="black", linewidth=1.8)
    ax.add_patch(rect)
    ax.text(x + 0.5, y + 0.68, short_name, ha="center", va="center", fontsize=18, fontweight="bold")
    ax.text(x + 0.5, y + 0.48, long_name, ha="center", va="center", fontsize=12)
    ax.text(x + 0.5, y + 0.25, str(count), ha="center", va="center", fontsize=18, fontweight="bold")

# Column labels.
ax.text(0.5, 2.13, "Predicted YES", ha="center", va="center", fontsize=13, fontweight="bold")
ax.text(1.5, 2.13, "Predicted NO", ha="center", va="center", fontsize=13, fontweight="bold")

# Row labels.
ax.text(-0.18, 1.5, "Actual YES", ha="right", va="center", fontsize=13, fontweight="bold")
ax.text(-0.18, 0.5, "Actual NO", ha="right", va="center", fontsize=13, fontweight="bold")

# Axis group labels.
ax.text(1.0, 2.42, "Confusion Matrix on Test Set", ha="center", va="center", fontsize=16, fontweight="bold")
ax.text(1.0, 2.27, "Columns = model prediction", ha="center", va="center", fontsize=11, color="gray")
ax.text(-0.48, 1.0, "Rows = true label", ha="center", va="center", fontsize=11, color="gray", rotation=90)

plt.tight_layout()
plt.show()
